# 03b. Validar lista de ouro

Censo da [`LISTA_OURO_ARQUIVO`](../config.py) contra `cpf_limpo` do estado
(CPFs da ouro **dentro**). JSON do [`02_treinar_splink.ipynb`](02_treinar_splink.ipynb),
11 regras e `predict(0,5)` como no [`02b_aplicar_splink.ipynb`](02b_aplicar_splink.ipynb),
atribuição como no [`04_atribuir.ipynb`](04_atribuir.ipynb).

Métricas e limites: [`README_validacao_ouro.md`](../README_validacao_ouro.md).
Não usa `*_aplicacao` nem `cohort_dedup` como rótulo. Predict grava
`splink_predictions_ouro.parquet` (não sobrescreve o 02b).
Gráficos Splink (`clerical_match_score`): ranking pairwise, não a associação 1:1.


In [ ]:
import json
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from IPython.display import display

from config import (
    DUCKDB_MEMORY_LIMIT,
    DUCKDB_THREADS,
    LISTA_OURO_ARQUIVO,
    MAX_CENSOS_POR_CPF,
    OUTPUT_DIR,
    PREDICT_NUM_CHUNKS_LEFT,
    PREDICT_NUM_CHUNKS_RIGHT,
    SPLINK_INPUT_VIEW,
    SPLINK_MODEL_JSON,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    cpf_norm_sql,
    drop_splink_temp_tables,
    ensure_output_dir,
    get_connection,
    get_splink_db_api,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T = THRESHOLD_AVALIACAO
TOP_N = 20
PREDICTIONS_OURO = OUTPUT_DIR / 'splink_predictions_ouro.parquet'

print_paths()
print('LISTA_OURO_ARQUIVO:', LISTA_OURO_ARQUIVO)
print('PREDICTIONS_OURO:', PREDICTIONS_OURO)
require_input(LISTA_OURO_ARQUIVO, label='LISTA_OURO')
if not SPLINK_MODEL_JSON.exists():
    raise RuntimeError(
        f'Modelo Splink não encontrado: {SPLINK_MODEL_JSON}. '
        'Rode notebooks/02_treinar_splink.ipynb — este notebook não retreina.'
    )

con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA],
    notebook_origem='00b',
)
print('T:', T, '| max Censos/CPF:', MAX_CENSOS_POR_CPF)
print(
    'DuckDB: threads / memory_limit =',
    con.execute("SELECT current_setting('threads'), current_setting('memory_limit')").fetchone(),
    f'(defaults {DUCKDB_THREADS}, {DUCKDB_MEMORY_LIMIT})',
)
print('Predict chunks L/R:', PREDICT_NUM_CHUNKS_LEFT, PREDICT_NUM_CHUNKS_RIGHT)


## Lista de ouro

`DESCRIBE` do parquet. Colunas desta rodada: `ID_MORADOR` (Censo) e `cpf_cpf`.
G = pares 1:1 com os dois ids no limpo.


In [ ]:
display(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{LISTA_OURO_ARQUIVO}')").df())

con.execute(f'''
CREATE OR REPLACE TABLE lista_ouro AS
SELECT
    CAST("ID_MORADOR" AS VARCHAR) AS id_censo,
    {cpf_norm_sql('"cpf_cpf"')} AS cpf_norm
FROM read_parquet('{LISTA_OURO_ARQUIVO}')
WHERE "ID_MORADOR" IS NOT NULL AND "cpf_cpf" IS NOT NULL
''')

display(con.execute('''
SELECT
    COUNT(*) AS n_pares,
    COUNT(DISTINCT id_censo) AS n_censo,
    COUNT(DISTINCT cpf_norm) AS n_cpf
FROM lista_ouro
''').df())


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE gt_lista AS
SELECT
    'censo_' || id_censo AS unique_id_censo,
    'cpf_' || cpf_norm AS unique_id_cpf,
    id_censo,
    cpf_norm
FROM lista_ouro
QUALIFY
    COUNT(*) OVER (PARTITION BY id_censo) = 1
    AND COUNT(*) OVER (PARTITION BY cpf_norm) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE gt_ouro AS
SELECT gt.*
FROM gt_lista gt
JOIN {TABELA_CENSO_LIMPA} c ON c.unique_id = gt.unique_id_censo
JOIN {TABELA_CPF_LIMPA} p ON p.unique_id = gt.unique_id_cpf
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_ouro AS
SELECT *
FROM {TABELA_CENSO_LIMPA}
WHERE unique_id IN (SELECT unique_id_censo FROM gt_ouro)
''')

n_lista = con.execute('SELECT COUNT(*) FROM lista_ouro').fetchone()[0]
n_1a1 = con.execute('SELECT COUNT(*) FROM gt_lista').fetchone()[0]
n_censo_limpo = con.execute(f'''
SELECT COUNT(*) FROM gt_lista g
JOIN {TABELA_CENSO_LIMPA} c ON c.unique_id = g.unique_id_censo
''').fetchone()[0]
n_cpf_limpo = con.execute(f'''
SELECT COUNT(*) FROM gt_lista g
JOIN {TABELA_CPF_LIMPA} p ON p.unique_id = g.unique_id_cpf
''').fetchone()[0]
n_ouro = con.execute('SELECT COUNT(*) FROM gt_ouro').fetchone()[0]
n_censo_ouro = con.execute('SELECT COUNT(*) FROM censo_ouro').fetchone()[0]

display(con.execute(f'''
SELECT
    {n_lista} AS n_lista,
    {n_1a1} AS n_1a1,
    {n_lista - n_1a1} AS n_nao_1a1,
    {n_censo_limpo} AS n_1a1_censo_no_limpo,
    {n_cpf_limpo} AS n_1a1_cpf_no_limpo,
    {n_ouro} AS n_ouro,
    {n_censo_ouro} AS n_censo_ouro
''').df())

if n_ouro == 0:
    raise RuntimeError(
        'Nenhum par 1:1 da lista caiu no limpo — confira o recorte do NB00.'
    )
print('G (n_ouro):', f'{n_ouro:,}')


Quadrantes no join Censo ouro × CPF ouro. Nulo não conta como igual.


In [ ]:
display(con.execute(f'''
SELECT
    CASE
        WHEN ca.nome_completo_phon IS NOT NULL
         AND pb.nome_completo_phon IS NOT NULL
         AND ca.nome_completo_phon = pb.nome_completo_phon
        THEN 'nome_igual'
        ELSE 'nome_diferente'
    END AS nome,
    CASE
        WHEN ca.data_nascimento IS NOT NULL
         AND pb.data_nascimento IS NOT NULL
         AND ca.data_nascimento = pb.data_nascimento
        THEN 'dob_igual'
        ELSE 'dob_diferente'
    END AS dob,
    COUNT(*) AS n
FROM gt_ouro gt
JOIN {TABELA_CENSO_LIMPA} ca ON ca.unique_id = gt.unique_id_censo
JOIN {TABELA_CPF_LIMPA} pb ON pb.unique_id = gt.unique_id_cpf
GROUP BY 1, 2
ORDER BY 1, 2
''').df())


## Blocking e predict

Onze regras OR do 02b. Gráfico cumulativo **antes** do predict. Se o volume
explodir, pare. Views: `censo_ouro` × `cpf_limpo`.


In [ ]:
from splink import block_on

blocking_rules = [
    block_on('nome_completo_phon'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento'),
    block_on('primeiro_nome_phon', 'ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento'),
    block_on('primeiro_nome_phon', 'data_nascimento'),
    block_on('ultimo_nome_phon', 'data_nascimento'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'dia_nascimento', 'cod_municipio'),
    block_on('primeiro_nome_phon', 'mes_nascimento', 'ano_nascimento', 'cod_municipio'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'dia_nascimento', 'sexo', 'cep'),
    block_on('ultimo_nome_phon', 'mes_nascimento', 'ano_nascimento', 'sexo', 'cep'),
    block_on('data_nascimento', 'sexo', 'cep'),
]


In [ ]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_data,
)
from splink.internals.charts import cumulative_blocking_rule_comparisons_generated

materialize_splink_input(
    con,
    censo_table='censo_ouro',
    cpf_table=TABELA_CPF_LIMPA,
)
db_api = get_splink_db_api(con)

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_chart = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_chart = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
print(f'Chart blocking: censo {n_censo_chart:,} | cpf {n_cpf_chart:,}')

df_blocking = cumulative_comparisons_to_be_scored_from_blocking_rules_data(
    table_or_tables=['splink_censo', 'splink_cpf'],
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type='link_only',
)
n_pares_blocking = int(df_blocking['cumulative_rows'].iloc[-1])
print(f'Pares OR: {n_pares_blocking:,}')
cumulative_blocking_rule_comparisons_generated(df_blocking.to_dict(orient='records'))


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')
n_censo_view = con.execute('SELECT COUNT(*) FROM splink_censo').fetchone()[0]
n_cpf_view = con.execute('SELECT COUNT(*) FROM splink_cpf').fetchone()[0]
n_censo_tab = con.execute('SELECT COUNT(*) FROM censo_ouro').fetchone()[0]
n_cpf_tab = con.execute(f'SELECT COUNT(*) FROM {TABELA_CPF_LIMPA}').fetchone()[0]
assert n_censo_view == n_censo_tab and n_cpf_view == n_cpf_tab, (
    f'Linker fora do recorte ouro × cpf_limpo: '
    f'censo {n_censo_view} vs {n_censo_tab}; cpf {n_cpf_view} vs {n_cpf_tab}'
)
print('Linker: censo_ouro', f'{n_censo_view:,}', '| cpf_limpo', f'{n_cpf_view:,}')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as file:
    data = json.load(file)
data['retain_intermediate_calculation_columns'] = False
data['retain_matching_columns'] = False
data['blocking_rules_to_generate_predictions'] = blocking_rules

linker = Linker(
    ['splink_censo', 'splink_cpf'],
    data,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)


In [ ]:
predict_kwargs = {'threshold_match_probability': 0.5}
if PREDICT_NUM_CHUNKS_LEFT is not None:
    predict_kwargs['num_chunks_left'] = PREDICT_NUM_CHUNKS_LEFT
if PREDICT_NUM_CHUNKS_RIGHT is not None:
    predict_kwargs['num_chunks_right'] = PREDICT_NUM_CHUNKS_RIGHT

df_predict = linker.inference.predict(**predict_kwargs)
pred = df_predict.physical_name
print('predictions table:', pred)
print('pares:', con.execute(f'SELECT COUNT(*) FROM {pred}').fetchone()[0])

ensure_output_dir()
cols = [r[0] for r in con.execute(f'DESCRIBE {pred}').fetchall()]
keep = [
    c for c in ('unique_id_l', 'unique_id_r', 'match_probability', 'match_weight')
    if c in cols
]
con.execute(
    f"COPY (SELECT {', '.join(keep)} FROM {pred}) "
    f"TO '{PREDICTIONS_OURO}' (FORMAT PARQUET, COMPRESSION ZSTD)"
)
drop_splink_temp_tables(con)
print('Predictions ouro:', PREDICTIONS_OURO)
print('colunas:', keep)


## Atribuição

Mesmo SQL do 04: `p ≥ T`, veto de mãe, melhor CPF por Censo, teto 3. Sem o 05.


In [ ]:
_tipo = con.execute('''
SELECT table_type
FROM information_schema.tables
WHERE table_schema = 'main' AND table_name = 'splink_predictions'
''').fetchone()
if _tipo:
    _kind = 'VIEW' if _tipo[0].upper() == 'VIEW' else 'TABLE'
    con.execute(f'DROP {_kind} IF EXISTS splink_predictions')
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{PREDICTIONS_OURO}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_por_censo AS
SELECT p.unique_id_censo, p.unique_id_cpf, p.match_probability
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
WHERE p.match_probability >= {T}
  AND (
        ca.nome_mae_phon IS NULL
        OR pb.nome_mae_phon IS NULL
        OR ca.nome_mae_phon = pb.nome_mae_phon
        OR (
            least(
                len(string_split(ca.nome_mae_phon, ' ')),
                len(string_split(pb.nome_mae_phon, ' '))
            ) >= 2
            AND list_slice(
                string_split(ca.nome_mae_phon, ' '),
                1,
                least(
                    len(string_split(ca.nome_mae_phon, ' ')),
                    len(string_split(pb.nome_mae_phon, ' '))
                )
            ) = list_slice(
                string_split(pb.nome_mae_phon, ' '),
                1,
                least(
                    len(string_split(ca.nome_mae_phon, ' ')),
                    len(string_split(pb.nome_mae_phon, ' '))
                )
            )
        )
        OR (
            ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL
            AND ca.nome_completo_phon = pb.nome_completo_phon
            AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
            AND ca.data_nascimento = pb.data_nascimento
        )
  )
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY p.unique_id_censo
    ORDER BY
        p.match_probability DESC,
        (
            CAST(ca.cep IS NOT NULL AND pb.cep IS NOT NULL AND ca.cep = pb.cep AS INTEGER)
            + CAST(
                ca.nome_mae_phon IS NOT NULL AND pb.nome_mae_phon IS NOT NULL
                AND (
                    ca.nome_mae_phon = pb.nome_mae_phon
                    OR (
                        least(
                            len(string_split(ca.nome_mae_phon, ' ')),
                            len(string_split(pb.nome_mae_phon, ' '))
                        ) >= 2
                        AND list_slice(
                            string_split(ca.nome_mae_phon, ' '),
                            1,
                            least(
                                len(string_split(ca.nome_mae_phon, ' ')),
                                len(string_split(pb.nome_mae_phon, ' '))
                            )
                        ) = list_slice(
                            string_split(pb.nome_mae_phon, ' '),
                            1,
                            least(
                                len(string_split(ca.nome_mae_phon, ' ')),
                                len(string_split(pb.nome_mae_phon, ' '))
                            )
                        )
                    )
                ) AS INTEGER
            )
        ) DESC,
        p.unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE associacoes_unicas AS
SELECT m.*
FROM melhor_por_censo m
JOIN (
    SELECT unique_id_cpf
    FROM melhor_por_censo
    GROUP BY 1
    HAVING COUNT(*) <= {MAX_CENSOS_POR_CPF}
) c ON c.unique_id_cpf = m.unique_id_cpf
''')

n_antes = con.execute(f'''
SELECT COUNT(DISTINCT unique_id_censo)
FROM splink_predictions
WHERE match_probability >= {T}
''').fetchone()[0]
n_melhor = con.execute('SELECT COUNT(*) FROM melhor_por_censo').fetchone()[0]
n_unicas = con.execute('SELECT COUNT(*) FROM associacoes_unicas').fetchone()[0]
print('Censos com par >= T (antes do veto de mãe):', f'{n_antes:,}')
print('Censos com par >= T (depois do veto):', f'{n_melhor:,}')
print('Associações (n_censo <=', MAX_CENSOS_POR_CPF, '):', f'{n_unicas:,}')


## Pacote de métricas

Cada Censo de G em um corte. FN = não `associou_cpf_ouro`. FP da associação =
errado entre quem associou. FP da escolha = `M ≠ Y` entre quem tem `melhor_por_censo`.


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE avaliacao_ouro AS
SELECT
    gt.unique_id_censo,
    gt.unique_id_cpf AS unique_id_cpf_ouro,
    a.unique_id_cpf AS unique_id_cpf_associado,
    m.unique_id_cpf AS unique_id_cpf_escolha,
    a.match_probability AS p_associado,
    m.match_probability AS p_escolha,
    CASE
        WHEN a.unique_id_cpf = gt.unique_id_cpf THEN 'associou_cpf_ouro'
        WHEN a.unique_id_censo IS NOT NULL THEN 'associou_cpf_errado'
        WHEN m.unique_id_censo IS NOT NULL THEN 'melhor_par_mas_cpf_repartilhado'
        WHEN p.unique_id_censo IS NOT NULL THEN 'teve_par_abaixo_de_T'
        ELSE 'nenhum_par'
    END AS corte
FROM gt_ouro gt
LEFT JOIN associacoes_unicas a ON a.unique_id_censo = gt.unique_id_censo
LEFT JOIN melhor_por_censo m ON m.unique_id_censo = gt.unique_id_censo
LEFT JOIN (
    SELECT DISTINCT unique_id_censo FROM splink_predictions
) p ON p.unique_id_censo = gt.unique_id_censo
''')

display(con.execute('''
SELECT corte, COUNT(*) AS n
FROM avaliacao_ouro
GROUP BY 1
ORDER BY 1
''').df())


In [ ]:
display(con.execute('''
SELECT
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro') AS n_associou_cpf_ouro,
    COUNT(*) FILTER (WHERE corte = 'associou_cpf_errado') AS n_associou_cpf_errado,
    COUNT(*) FILTER (WHERE corte = 'melhor_par_mas_cpf_repartilhado')
        AS n_melhor_par_mas_cpf_repartilhado,
    COUNT(*) FILTER (WHERE corte = 'teve_par_abaixo_de_T') AS n_teve_par_abaixo_de_T,
    COUNT(*) FILTER (WHERE corte = 'nenhum_par') AS n_nenhum_par,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_pct,
    ROUND(
        100.0 * (
            COUNT(*) - COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS fn_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_ouro')
        / NULLIF(
            COUNT(*) FILTER (WHERE corte IN ('associou_cpf_ouro', 'associou_cpf_errado')),
            0
        ),
        2
    ) AS precisao_associacao_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE corte = 'associou_cpf_errado')
        / NULLIF(
            COUNT(*) FILTER (WHERE corte IN ('associou_cpf_ouro', 'associou_cpf_errado')),
            0
        ),
        2
    ) AS fp_associacao_pct
FROM avaliacao_ouro
''').df())


In [ ]:
display(con.execute('''
SELECT
    COUNT(*) AS n_com_escolha,
    COUNT(*) FILTER (WHERE unique_id_cpf_escolha <> unique_id_cpf_ouro)
        AS n_escolha_errada,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE unique_id_cpf_escolha <> unique_id_cpf_ouro)
        / NULLIF(COUNT(*), 0),
        2
    ) AS fp_escolha_pct
FROM avaliacao_ouro
WHERE unique_id_cpf_escolha IS NOT NULL
''').df())

display(con.execute('''
SELECT
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE p.unique_id_censo IS NOT NULL) AS n_par_ouro_no_predict,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE p.unique_id_censo IS NOT NULL)
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_par_ouro_predict_pct
FROM gt_ouro gt
LEFT JOIN splink_predictions p
    ON p.unique_id_censo = gt.unique_id_censo
   AND p.unique_id_cpf = gt.unique_id_cpf
''').df())


Mesmo pacote nos quadrantes nome × DOB.


In [ ]:
display(con.execute(f'''
SELECT
    CASE
        WHEN ca.nome_completo_phon IS NOT NULL
         AND pb.nome_completo_phon IS NOT NULL
         AND ca.nome_completo_phon = pb.nome_completo_phon
        THEN 'nome_igual'
        ELSE 'nome_diferente'
    END AS nome,
    CASE
        WHEN ca.data_nascimento IS NOT NULL
         AND pb.data_nascimento IS NOT NULL
         AND ca.data_nascimento = pb.data_nascimento
        THEN 'dob_igual'
        ELSE 'dob_diferente'
    END AS dob,
    COUNT(*) AS n_ouro,
    COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_ouro') AS n_associou_cpf_ouro,
    COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_errado') AS n_associou_cpf_errado,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_ouro')
        / NULLIF(COUNT(*), 0),
        2
    ) AS recall_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE av.corte = 'associou_cpf_errado')
        / NULLIF(
            COUNT(*) FILTER (
                WHERE av.corte IN ('associou_cpf_ouro', 'associou_cpf_errado')
            ),
            0
        ),
        2
    ) AS fp_associacao_pct
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_ouro
GROUP BY 1, 2
ORDER BY 1, 2
''').df())


## Gráficos Splink (`clerical_match_score`)

Ranking **pairwise**, não a associação 1:1 do 04. Precisão daqui ≠ `associou_cpf_ouro / (ouro + errado)`.

Rótulo: par ouro = 1; outro CPF do mesmo Censo no predict = 0. Sem negativo a ROC não existe. Segundo Linker só pontua esses pares (`retain_*` ligado).

FP/FN do `prediction_errors` são do par (outro candidato com `p ≥ T`), não de `associou_cpf_errado`.


In [ ]:
con.execute('''
CREATE OR REPLACE TABLE labels_clericais AS
SELECT
    'censo' AS source_dataset_l,
    unique_id_censo AS unique_id_l,
    'cpf' AS source_dataset_r,
    unique_id_cpf AS unique_id_r,
    1.0 AS clerical_match_score
FROM gt_ouro
UNION ALL
SELECT
    'censo',
    p.unique_id_censo,
    'cpf',
    p.unique_id_cpf,
    0.0
FROM splink_predictions p
JOIN gt_ouro gt ON gt.unique_id_censo = p.unique_id_censo
WHERE p.unique_id_cpf <> gt.unique_id_cpf
''')

display(con.execute('''
SELECT
    clerical_match_score,
    COUNT(*) AS n
FROM labels_clericais
GROUP BY 1
ORDER BY 1 DESC
''').df())


In [ ]:
from splink import Linker

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as file:
    data_cl = json.load(file)
data_cl['retain_intermediate_calculation_columns'] = True
data_cl['retain_matching_columns'] = True
data_cl['blocking_rules_to_generate_predictions'] = blocking_rules

drop_splink_temp_tables(con)
db_api = get_splink_db_api(con)
linker_cl = Linker(
    ['splink_censo', 'splink_cpf'],
    data_cl,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)
print('Linker clerical: censo_ouro × cpf_limpo (retain ligado; não faz predict cheio)')


In [ ]:
from splink.internals.charts import (
    precision_recall_chart,
    roc_chart,
    threshold_selection_tool,
)

tab = linker_cl.evaluation.accuracy_analysis_from_labels_table(
    'labels_clericais',
    threshold_match_probability=0.5,
    match_weight_round_to_nearest=0.1,
    output_type='table',
    add_metrics=['f1'],
)
recs = tab.as_record_dict()
if not recs:
    raise RuntimeError(
        'Truth table vazia. Confira source_dataset em labels_clericais '
        '(esperado censo / cpf) contra o concat do Linker.'
    )

recs_tool = [d for d in recs if d['precision'] > 0.5 and d['recall'] > 0.5]
if recs_tool:
    display(threshold_selection_tool(recs, add_metrics=['f1']))
else:
    print('Threshold tool sem pontos com precisão e recall > 0,5; ROC e PR abaixo.')
display(roc_chart(recs))
display(precision_recall_chart(recs))

df_truth = tab.as_pandas_dataframe()
i = (df_truth['match_probability'] - T).abs().idxmin()
print('Linha mais perto de T =', T, '(precisão/recall pairwise, não da associação)')
display(
    df_truth.loc[
        [i],
        [
            'truth_threshold',
            'match_probability',
            'tp',
            'fp',
            'fn',
            'tn',
            'precision',
            'recall',
            'f1',
        ],
    ]
)


In [ ]:
erros = linker_cl.evaluation.prediction_errors_from_labels_table(
    'labels_clericais',
    include_false_positives=True,
    include_false_negatives=True,
    threshold_match_probability=T,
)
tab_erros = erros.physical_name
print('prediction_errors:', tab_erros, '| corte T =', T)
display(con.execute(f'''
SELECT truth_status, COUNT(*) AS n
FROM {tab_erros}
GROUP BY 1
ORDER BY 1
''').df())


In [ ]:
fn = con.execute(f'''
SELECT *
FROM {tab_erros}
WHERE truth_status = 'FN'
ORDER BY match_probability
LIMIT {TOP_N}
''').df()
display(fn[[
    'unique_id_l', 'unique_id_r', 'match_probability',
    'clerical_match_score', 'truth_status',
]])
if fn.empty:
    print('Nenhum FN pairwise em T.')
else:
    display(linker_cl.visualisations.waterfall_chart(fn.to_dict(orient='records')))


In [ ]:
fp = con.execute(f'''
SELECT *
FROM {tab_erros}
WHERE truth_status = 'FP'
ORDER BY match_probability DESC
LIMIT {TOP_N}
''').df()
print('FP pairwise (não é associou_cpf_errado)')
display(con.execute(f'''
SELECT
    e.unique_id_l,
    e.unique_id_r,
    gt.unique_id_cpf AS unique_id_cpf_ouro,
    e.match_probability,
    e.clerical_match_score,
    e.truth_status,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ouro.data_nascimento AS dob_cpf_ouro
FROM {tab_erros} e
JOIN gt_ouro gt ON gt.unique_id_censo = e.unique_id_l
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = e.unique_id_l
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = e.unique_id_r
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = gt.unique_id_cpf
WHERE e.truth_status = 'FP'
ORDER BY e.match_probability DESC, e.unique_id_l
LIMIT {TOP_N}
''').df())
if fp.empty:
    print('Nenhum FP pairwise em T.')
else:
    display(linker_cl.visualisations.waterfall_chart(fp.to_dict(orient='records')))


## Pares associados ao CPF errado

`A ≠ Y`: o 04 publicou **outro** CPF, não “não é a mesma pessoa”.
Ordenado por `p` desc — o topo parece acerto. Flags: nulo não conta como igual.


In [ ]:
cols_exemplo = '''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ouro.data_nascimento AS dob_cpf_ouro,
    ca.nome_mae AS mae_censo,
    pb.nome_mae AS mae_cpf,
    ouro.nome_mae AS mae_cpf_ouro,
    ca.cep AS cep_censo,
    pb.cep AS cep_cpf,
    ouro.cep AS cep_cpf_ouro
'''

display(con.execute(f'''
SELECT
    av.unique_id_censo,
    av.unique_id_cpf_associado,
    av.unique_id_cpf_ouro,
    av.p_associado,
    po.match_probability AS p_ouro,
    CAST(
        ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
        AND ca.nome_completo = pb.nome_completo
        AS INTEGER
    ) AS nome_assoc,
    CAST(
        ca.nome_completo IS NOT NULL AND ouro.nome_completo IS NOT NULL
        AND ca.nome_completo = ouro.nome_completo
        AS INTEGER
    ) AS nome_ouro,
    CAST(
        ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
        AND ca.data_nascimento = pb.data_nascimento
        AS INTEGER
    ) AS dob_assoc,
    CAST(
        ca.data_nascimento IS NOT NULL AND ouro.data_nascimento IS NOT NULL
        AND ca.data_nascimento = ouro.data_nascimento
        AS INTEGER
    ) AS dob_ouro,
    CAST(
        ca.nome_mae IS NOT NULL AND pb.nome_mae IS NOT NULL
        AND ca.nome_mae = pb.nome_mae
        AS INTEGER
    ) AS mae_assoc,
    CAST(
        ca.nome_mae IS NOT NULL AND ouro.nome_mae IS NOT NULL
        AND ca.nome_mae = ouro.nome_mae
        AS INTEGER
    ) AS mae_ouro,
    {cols_exemplo}
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_associado
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
LEFT JOIN splink_predictions po
    ON po.unique_id_censo = av.unique_id_censo
   AND po.unique_id_cpf = av.unique_id_cpf_ouro
WHERE av.corte = 'associou_cpf_errado'
ORDER BY av.p_associado DESC, av.unique_id_censo
LIMIT {TOP_N}
''').df())

display(con.execute(f'''
SELECT
    COUNT(*) AS n_associou_cpf_errado,
    COUNT(*) FILTER (
        WHERE ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
          AND ca.nome_completo = pb.nome_completo
          AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
          AND NOT (
              ouro.nome_completo IS NOT NULL
              AND ca.nome_completo = ouro.nome_completo
              AND ouro.data_nascimento IS NOT NULL
              AND ca.data_nascimento = ouro.data_nascimento
          )
    ) AS n_assoc_casa_ouro_nao,
    COUNT(*) FILTER (
        WHERE ca.nome_completo IS NOT NULL AND pb.nome_completo IS NOT NULL
          AND ca.nome_completo = pb.nome_completo
          AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL
          AND ca.data_nascimento = pb.data_nascimento
          AND ouro.nome_completo IS NOT NULL
          AND ca.nome_completo = ouro.nome_completo
          AND ouro.data_nascimento IS NOT NULL
          AND ca.data_nascimento = ouro.data_nascimento
    ) AS n_ambos_casam,
    COUNT(*) FILTER (
        WHERE po.match_probability IS NOT NULL
          AND po.match_probability >= av.p_associado
    ) AS n_p_ouro_ge_p_assoc
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = av.unique_id_cpf_associado
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
LEFT JOIN splink_predictions po
    ON po.unique_id_censo = av.unique_id_censo
   AND po.unique_id_cpf = av.unique_id_cpf_ouro
WHERE av.corte = 'associou_cpf_errado'
''').df())


Amostra de FN por corte (tudo que não associou o CPF ouro).


In [ ]:
display(con.execute(f'''
SELECT
    av.corte,
    av.unique_id_censo,
    av.unique_id_cpf_ouro,
    av.unique_id_cpf_associado,
    av.unique_id_cpf_escolha,
    av.p_associado,
    av.p_escolha,
    ca.nome_completo AS nome_censo,
    ouro.nome_completo AS nome_cpf_ouro,
    ca.data_nascimento AS dob_censo,
    ouro.data_nascimento AS dob_cpf_ouro
FROM avaliacao_ouro av
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = av.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ouro ON ouro.unique_id = av.unique_id_cpf_ouro
WHERE av.corte <> 'associou_cpf_ouro'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY av.corte
    ORDER BY av.unique_id_censo
) <= {TOP_N}
ORDER BY av.corte, av.unique_id_censo
''').df())


## Waterfall de um par

Cola os `unique_id` (os de `splink_input`). Reroda só esta célula. Linker no
conjunto ouro × `cpf_limpo` — o TF é o desse recorte. Blocking só o par.


In [ ]:
from splink import Linker

UNIQUE_ID_CENSO = 'censo_COLOQUE_O_ID'
UNIQUE_ID_CPF = 'cpf_COLOQUE_O_ID'

n_censo_id = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}
WHERE unique_id = '{UNIQUE_ID_CENSO}' AND origem = 'censo'
''').fetchone()[0]
n_cpf_id = con.execute(f'''
SELECT COUNT(*) FROM {SPLINK_INPUT_VIEW}
WHERE unique_id = '{UNIQUE_ID_CPF}' AND origem = 'cpf'
''').fetchone()[0]
if n_censo_id != 1 or n_cpf_id != 1:
    raise RuntimeError(
        f'Ids não encontrados em splink_input: '
        f'censo {UNIQUE_ID_CENSO!r} n={n_censo_id}; '
        f'cpf {UNIQUE_ID_CPF!r} n={n_cpf_id}'
    )

con.execute(f'''
CREATE OR REPLACE VIEW splink_censo AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'censo'
''')
con.execute(f'''
CREATE OR REPLACE VIEW splink_cpf AS
SELECT * FROM {SPLINK_INPUT_VIEW} WHERE origem = 'cpf'
''')

with open(SPLINK_MODEL_JSON, encoding='utf-8') as f:
    settings = json.load(f)
settings['retain_intermediate_calculation_columns'] = True
settings['retain_matching_columns'] = True
settings['blocking_rules_to_generate_predictions'] = [
    {
        'blocking_rule': (
            f'l."unique_id" = \'{UNIQUE_ID_CENSO}\' '
            f'AND r."unique_id" = \'{UNIQUE_ID_CPF}\''
        ),
        'sql_dialect': 'duckdb',
    }
]

drop_splink_temp_tables(con)
db_api = get_splink_db_api(con)
linker_wf = Linker(
    ['splink_censo', 'splink_cpf'],
    settings,
    db_api=db_api,
    input_table_aliases=['censo', 'cpf'],
)
df_wf = linker_wf.inference.predict()
n_wf = con.execute(f'SELECT COUNT(*) FROM {df_wf.physical_name}').fetchone()[0]
if n_wf != 1:
    raise RuntimeError(f'Esperava 1 par no waterfall, veio {n_wf}')
row_wf = con.execute(f'''
SELECT match_probability, match_weight
FROM {df_wf.physical_name}
''').fetchone()
print('match_probability:', row_wf[0])
print('match_weight:', row_wf[1])

records = df_wf.as_record_dict(limit=1)
chart = linker_wf.visualisations.waterfall_chart(records)
drop_splink_temp_tables(con)
chart
